# 4. Catalogue, provenance et contrôles

1. **Jeux de données** : titre, version, DOI, résolution, couverture, licence et limites à afficher.
2. **Indicateurs** et **couverture réelle** des séries, règles de calcul.
3. **Contrôles automatiques** du pipeline (`validation.json`).
4. **Fichiers publiés** dans `data/api/`.
5. **Livrable P2 du backend** (`data/regions.geojson`, `data/processed/`) : forme, périodes, part des mois à faible couverture.
6. **Contrôle LST Terra / Aqua** (étape `control-lst`), s'il a été lancé.

In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as w
from IPython.display import JSON, Markdown, display

import agbea_nb as nb

pd.set_option("display.max_rows", 60)
cat = nb.catalog()
print(f"Données : {nb.API}")
print(f"Générées le {cat['generated_at']} · pipeline {cat['pipeline_version']} · référence {cat['baseline']}")

## 1. Jeux de données

In [ ]:
ds = pd.DataFrame(cat["datasets"]).T
display(ds[["title", "short_name", "version", "provider", "doi", "native_resolution", "temporal_resolution",
            "coverage", "license"]])
for k, d in cat["datasets"].items():
    display(Markdown(f"**{k}** — {d['title']} ([lien]({d['url']}))\n" + "\n".join(f"- {l}" for l in d.get("limits", []))))

## 2. Indicateurs, couverture et règles

In [ ]:
ind = pd.DataFrame(cat["indicators"]).T
cov = pd.DataFrame(cat["coverage"]).T
display(ind.join(cov, how="left"))
display(Markdown("**Nature** : " + " · ".join(f"`{k}` = {v}" for k, v in cat["nature_legend"].items())))
print(json.dumps({k: cat[k] for k in ("baseline", "seasons", "completeness", "trend_test")}, ensure_ascii=False, indent=1))

## 3. Contrôles automatiques (`validation.json`)

In [ ]:
val = json.loads((nb.API / "validation.json").read_text(encoding="utf-8"))
print(val["generated_at"], val["summary"])
couleur = {"pass": "background-color:#d8f0d8", "warn": "background-color:#fff1c2", "fail": "background-color:#f6c6c6"}
display(pd.DataFrame(val["checks"]).style.map(lambda s: couleur.get(s, ""), subset=["status"]))

## 4. Fichiers publiés dans `data/api/`

In [ ]:
rows = []
for p in sorted(nb.API.rglob("*")):
    if p.is_file():
        rel = p.relative_to(nb.API)
        rows.append({"dossier": str(rel.parent), "fichier": rel.name, "octets": p.stat().st_size})
f = pd.DataFrame(rows)
par_dossier = f.groupby("dossier").agg(fichiers=("fichier", "count"), taille=("octets", "sum"))
par_dossier["taille"] = par_dossier.taille.map(nb.human_size)
display(par_dossier)
print("Total :", len(f), "fichiers,", nb.human_size(int(f.octets.sum())))

## 5. Livrable P2 du backend

Ce que l'API lit réellement. `value` y est la moyenne des pixels visibles **quelle que soit la couverture** ;
`low_coverage = true` (moins de 50 % de la région visible) doit être signalé à l'utilisateur.

In [ ]:
reg = json.loads((nb.DATA / "regions.geojson").read_text(encoding="utf-8"))
display(pd.DataFrame([f["properties"] for f in reg["features"]]))

p2 = pd.read_parquet(nb.DATA / "processed" / "monthly.parquet")
display(p2.groupby(["indicator", "dataset_id"]).agg(premier=("date", "min"), dernier=("date", "max"), lignes=("value", "size"),
                                                   valeurs_nulles=("value", lambda v: int(v.isna().sum())),
                                                   faible_couverture_pct=("low_coverage", lambda v: round(100 * v.mean(), 1))))

static = json.loads((nb.DATA / "processed" / "static.json").read_text(encoding="utf-8"))
print("static.json :", static.get("dataset_id"), static.get("units"))
display(pd.DataFrame({k: v["elevation"] for k, v in static["regions"].items()}).T)

In [ ]:
def faible_couverture(indicator="ndvi"):
    d = p2[p2.indicator == indicator].assign(mois=lambda x: x.date.str[5:7].astype(int))
    h = d.groupby(["region_id", "mois"]).low_coverage.mean().unstack() * 100
    fig = go.Figure(go.Heatmap(z=h.values, x=h.columns, y=h.index, colorscale="Reds", zmin=0, zmax=100,
                               colorbar=dict(title="% des années"),
                               hovertemplate="%{y}, mois %{x} : %{z:.0f} %<extra></extra>"))
    fig.update_layout(title=f"{indicator} : part des années où le mois est à faible couverture (< 50 % visible)",
                      height=330, xaxis=dict(dtick=1, title="mois"))
    fig.show()

w.interact(faible_couverture, indicator=w.Dropdown(options=sorted(p2.indicator.unique()), value="ndvi"));

## 6. Contrôle des tendances LST : Terra contre Aqua

Décision D25 (révisée) : Terra et Aqua sont comparés sur les **mêmes mois**. Une tendance n'est **robuste**
que sur la **période de décision 2003-2020** (avant les dérives d'orbite des deux satellites), si les deux la
trouvent significative et de même signe. La période 2003-2025 n'est qu'un diagnostic (`robust` vide).
L'écart Terra − Aqua des anomalies est testé à part : une tendance de cet écart signale une dérive.
Fichiers produits par `docker compose run --rm pipeline control-lst`.

In [ ]:
ctrl = nb.DATA / "interim" / "work" / "controls"
if not (ctrl / "lst_terra_aqua.parquet").exists():
    print("Contrôle pas encore lancé : docker compose run --rm pipeline lst-aqua, puis control-lst")
else:
    c = pd.read_parquet(ctrl / "lst_terra_aqua.parquet")
    g = c.groupby(["variable", "season", "period"])
    display(Markdown("**Tendances par satellite (toutes unités)**"))
    display(pd.DataFrame({
        "unités": g.size(),
        "Terra significatif %": g.significant_terra.mean().mul(100).round(0),
        "Aqua significatif %": g.significant_aqua.mean().mul(100).round(0),
        "même signe %": g.same_sign.mean().mul(100).round(0),
        "robuste % (2003-2020)": g.robust.apply(lambda r: r.dropna().astype(bool).mean() * 100 if r.notna().any() else np.nan).round(0),
        "pente Terra médiane": g.sen_slope_per_decade_terra.median().round(2),
        "pente Aqua médiane": g.sen_slope_per_decade_aqua.median().round(2),
    }))
    gap = pd.read_parquet(ctrl / "lst_terra_aqua_ecart.parquet")
    gg = gap.groupby(["variable", "period"])
    display(Markdown("**Écart Terra − Aqua : tendance de l'écart des anomalies (signal de dérive)**"))
    display(pd.DataFrame({"unités": gg.size(), "écart significatif %": gg.gap_significant.mean().mul(100).round(0),
                          "pente médiane de l'écart (°C / décennie)": gg.gap_slope_per_decade.median().round(2)}))
    r = c[c.level.isin(["region", "site"]) & (c.season == "annual")]
    display(r[["name", "level", "variable", "period", "sen_slope_per_decade_terra", "p_value_terra",
               "sen_slope_per_decade_aqua", "p_value_aqua", "robust"]]
            .sort_values(["variable", "period", "level", "name"]).round(3).reset_index(drop=True))